# Camera Discovery — HLS Harvest Handoff to Full Validation

This notebook tests the end-to-end path where HLS-only harvest output seeds the normal `camera-discovery run` pipeline. It verifies that `harvest_handoff.json` remains media-filter aware, so `.m3u8` harvests stay HLS-only when passed into the pipeline.

This notebook uses `--profile full`, which can be slower and may perform media validation. Trusted output still requires existing validation, scope, and trust gates; coordinates alone do not make a candidate trusted.


The run-from-handoff section uses `--harvest-input-mode handoff-only`, so native discovery is disabled and the candidate set remains bounded by the selected handoff records. Use `--harvest-input-mode seed` only when intentionally combining the handoff with normal discovery.


## Setup

This notebook installs the repository code and runs the public CLI. It does not patch source files from the notebook.

Default behavior clones the `dev` branch. In Colab, restart the runtime after dependency installation if package imports behave unexpectedly.


In [ ]:
# Repository setup for Google Colab / notebook execution.
# Change REPO_BRANCH or REPO_URL if testing a fork/PR branch.
REPO_BRANCH = "dev"
REPO_URL = "https://github.com/dshipley71/camera-discovery.git"
REPO_DIR = "/content/camera-discovery"

from pathlib import Path
repo_dir = Path(REPO_DIR)
if not repo_dir.exists():
    !git clone -b "{REPO_BRANCH}" "{REPO_URL}" "{REPO_DIR}"
else:
    print(f"Repository already exists at {repo_dir}. Keeping existing checkout.")
%cd {REPO_DIR}
%pip install -e .[cloakbrowser] --no-build-isolation


## Credentials

In [1]:
# Ollama Cloud / LLM credential setup.
# This avoids printing secrets. Configure OLLAMA_API_KEY in Colab: left sidebar > Secrets.
import os

try:
    from google.colab import userdata  # type: ignore
    OLLAMA_API_KEY = userdata.get('OLLAMA_API_KEY')
except Exception:
    OLLAMA_API_KEY = os.environ.get('OLLAMA_API_KEY')

if OLLAMA_API_KEY:
    os.environ['OLLAMA_API_KEY'] = OLLAMA_API_KEY
    os.environ.setdefault('CAMERA_DISCOVERY_LLM_PROVIDER', 'ollama-cloud')
    os.environ.setdefault('CAMERA_DISCOVERY_LLM_MODEL', 'gemma3:27b-cloud')
    os.environ.setdefault('CAMERA_DISCOVERY_TARGET_INTENT_MODEL', 'gemma3:12b-cloud')
    os.environ.setdefault('CAMERA_DISCOVERY_TARGET_INTENT_FALLBACK_MODEL', 'gemma3:12b-cloud')
    os.environ.setdefault('CAMERA_DISCOVERY_TARGET_INTENT_MODEL', 'gemma3:12b-cloud')
    os.environ.setdefault('CAMERA_DISCOVERY_GEOCODER_REFEREE_MODEL', 'gemma3:27b-cloud')
    os.environ.setdefault('CAMERA_DISCOVERY_LOCATION_INFERENCE_MODEL', 'gemma3:27b-cloud')
    print('Loaded OLLAMA_API_KEY from Colab userdata/environment')
else:
    print('OLLAMA_API_KEY not found. LLM-backed stages may fail unless another provider is configured.')

# Keep these visible so output records the provider/model, but never print the key.
print('LLM provider:', os.environ.get('CAMERA_DISCOVERY_LLM_PROVIDER', '(default from config)'))
print('LLM model:', os.environ.get('CAMERA_DISCOVERY_LLM_MODEL', '(default from config)'))


Loaded OLLAMA_API_KEY from Colab userdata/environment
LLM provider: ollama-cloud
LLM model: gemma3:27b-cloud


## Smoke tests

In [ ]:
# CLI and public import smoke tests.
import subprocess, sys

def run_cmd(cmd, *, env=None, check=True):
    print("\n$", " ".join(str(part) for part in cmd))
    result = subprocess.run([str(part) for part in cmd], env=env, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {' '.join(map(str, cmd))}")
    return result

run_cmd(['camera-discovery', '--help'])
run_cmd(['camera-discovery', 'run', '--help'])
run_cmd(['camera-discovery', 'harvest-urls', '--help'])

from camera_discovery.services.discovery_engine import CandidateDiscoveryEngine
from camera_discovery.services.harvest_engine import CameraUrlHarvestEngine
import camera_discovery.cli
print('camera-discovery imports OK')


## Notebook-only helper functions

In [2]:
# Notebook-only inspection helpers. These intentionally live in the notebook, not src/.
from __future__ import annotations

import json
import os
import shutil
import subprocess
from collections import Counter
from pathlib import Path
from urllib.parse import urlparse


def read_json(path):
    path = Path(path)
    if not path.exists():
        print(f"Missing: {path}")
        return None
    return json.loads(path.read_text(encoding='utf-8'))


def iter_jsonl(path, limit=None):
    path = Path(path)
    if not path.exists():
        return
    with path.open('r', encoding='utf-8') as f:
        for idx, line in enumerate(f):
            if limit is not None and idx >= limit:
                break
            line = line.strip()
            if not line:
                continue
            yield json.loads(line)


def count_jsonl(path):
    path = Path(path)
    if not path.exists():
        return 0
    with path.open('r', encoding='utf-8') as f:
        return sum(1 for line in f if line.strip())


def top_hosts(path, url_field='url', limit=15):
    counts = Counter()
    for row in iter_jsonl(path):
        url = row.get(url_field) or row.get('stream_url') or row.get('source_url') or ''
        host = urlparse(url).netloc.casefold() or '(missing-host)'
        counts[host] += 1
    return counts.most_common(limit)


def media_counts(path):
    counts = Counter()
    for row in iter_jsonl(path):
        counts[row.get('media_type') or row.get('camera_type') or '(missing)'] += 1
    return dict(counts)


def scope_counts(path):
    counts = Counter()
    for row in iter_jsonl(path):
        counts[row.get('scope_status') or row.get('properties', {}).get('scope_status') or '(missing)'] += 1
    return dict(counts)


def print_json(path, keys=None):
    data = read_json(path)
    if data is None:
        return None
    if keys:
        data = {key: data.get(key) for key in keys}
    print(json.dumps(data, indent=2, sort_keys=True)[:12000])
    return data


def list_existing(paths):
    for path in paths:
        path = Path(path)
        print(f"{path}: {'exists' if path.exists() else 'missing'}" + (f" ({path.stat().st_size:,} bytes)" if path.exists() and path.is_file() else ''))


def package_output(output_dir, zip_name=None):
    output_dir = Path(output_dir)
    if zip_name is None:
        zip_name = str(output_dir).rstrip('/').replace('/', '_') + '.zip'
    zip_base = Path(zip_name).with_suffix('')
    archive = shutil.make_archive(str(zip_base), 'zip', root_dir=str(output_dir))
    print('Created archive:', archive)
    try:
        from google.colab import files  # type: ignore
        files.download(archive)
    except Exception:
        print('Download helper unavailable outside Colab. Archive remains at:', archive)
    return archive



## Browser backend behavior

These notebooks default to browser capture disabled for structured endpoint / HLS workflows because the useful camera records usually come from static pages and JSON endpoints. Enable browser capture only when testing dynamic pages.

The cells below make the selected backend visible. They do not fake browser success.


In [3]:
# Browser backend configuration for this notebook.
# For routine HLS/structured-endpoint tests, keep browser capture disabled.
BROWSER_BACKEND = "playwright"  # change to "cloakbrowser" when intentionally testing that backend
DISABLE_BROWSER_CAPTURE_ENV = {
    "CAMERA_DISCOVERY_ENABLE_BROWSER_CAPTURE": "false",
    "CAMERA_DISCOVERY_BROWSER_BACKEND": BROWSER_BACKEND,
}
os.environ.update(DISABLE_BROWSER_CAPTURE_ENV)

print('Browser backend selected:', BROWSER_BACKEND)
print('Browser capture default for this notebook:', os.environ['CAMERA_DISCOVERY_ENABLE_BROWSER_CAPTURE'])
print('Browser backend exported to shell commands:', os.environ['CAMERA_DISCOVERY_BROWSER_BACKEND'])
print('To test dynamic browser capture, remove --disable-browser-capture for harvest and set CAMERA_DISCOVERY_ENABLE_BROWSER_CAPTURE=true for run.')


Browser backend selected: playwright
Browser capture default for this notebook: false
Browser backend exported to shell commands: playwright
To test dynamic browser capture, remove --disable-browser-capture for harvest and set CAMERA_DISCOVERY_ENABLE_BROWSER_CAPTURE=true for run.


## Step 1 — Harvest HLS-only URLs

In [4]:
%%time
QUERY = "Virginia traffic cameras"
HARVEST_DIR = Path('runs/harvest-california-hls')
RERUN_HARVEST = False

expected = [HARVEST_DIR / 'harvest_summary.json', HARVEST_DIR / 'harvest_handoff.json', HARVEST_DIR / 'camera_urls.jsonl']
if RERUN_HARVEST and HARVEST_DIR.exists():
    shutil.rmtree(HARVEST_DIR)

!camera-discovery harvest-urls "Virginia traffic cameras" \
  --output-dir runs/harvest-california-hls \
  --discovery-mode both \
  --max-search-queries 12 \
  --max-search-results-per-query 25 \
  --max-source-rows 2000 \
  --max-pages-per-source 10 \
  --max-urls 0 \
  --media .m3u8 \
  --disable-browser-capture \
  --progress-style plain



Harvest mode: extraction-only raw camera/media URL harvesting
Bypasses: target resolution, geocoding, validation, trust, scope, LLM review, 
GeoJSON, maps, cameras.md, review ZIP
Discovery mode: both
Sources file: /content/camera-discovery/SOURCES.md
Media filter: .m3u8
Intermediate records: disabled
Image asset filter: raw
Progress: harvest started — discovery_mode=both
Progress: harvest source rows ready — 215 rows (directory/SOURCES.md=46, 
blind=169, direct=0; sources_file_used=True; 
sources_file=/content/camera-discovery/SOURCES.md).
Progress: harvest rows 13/215; raw records=115.
Progress: harvest rows 16/215; raw records=220.
Progress: harvest rows 29/215; raw records=325.
Progress: harvest rows 31/215; raw records=426.
Progress: harvest rows 35/215; raw records=571.
Progress: harvest rows 39/215; raw records=699.
Progress: harvest rows 43/215; raw records=827.
Progress: harvest rows 46/215; raw records=1026.
Progress: harvest rows 61/215; raw records=1887.
Progress: harvest ro

## Inspect HLS harvest and handoff

In [ ]:
# Inspect harvest outputs.
HARVEST_DIR = Path(HARVEST_DIR)
print('Harvest directory:', HARVEST_DIR)
list_existing([
    HARVEST_DIR / 'harvest_summary.json',
    HARVEST_DIR / 'harvest_handoff.json',
    HARVEST_DIR / 'camera_urls.txt',
    HARVEST_DIR / 'camera_urls.csv',
    HARVEST_DIR / 'camera_urls.jsonl',
    HARVEST_DIR / 'camera_records.jsonl',
    HARVEST_DIR / 'camera_media_assets.jsonl',
    HARVEST_DIR / 'discovered_endpoints.jsonl',
    HARVEST_DIR / 'logs' / 'source_rows_summary.json',
    HARVEST_DIR / 'logs' / 'harvest_blind_search_diagnostics.jsonl',
    HARVEST_DIR / 'logs' / 'harvest_errors.jsonl',
])

summary = print_json(HARVEST_DIR / 'harvest_summary.json', keys=[
    'raw_count', 'unique_count', 'written_count', 'by_media_type', 'by_source_provider', 'by_source_host',
    'camera_record_count', 'media_asset_count', 'discovered_endpoint_count', 'warnings'
])
print('\nSource row summary:')
print_json(HARVEST_DIR / 'logs' / 'source_rows_summary.json')
print('\nHandoff manifest:')
print_json(HARVEST_DIR / 'harvest_handoff.json')
print('\nMedia counts from camera_urls.jsonl:', media_counts(HARVEST_DIR / 'camera_urls.jsonl'))
print('\nTop URL hosts from camera_urls.jsonl:', top_hosts(HARVEST_DIR / 'camera_urls.jsonl'))
print('\nJSONL counts:')
for name in ['camera_urls.jsonl', 'camera_records.jsonl', 'camera_media_assets.jsonl', 'discovered_endpoints.jsonl', 'harvest_camera_inventory.jsonl']:
    print(name, count_jsonl(HARVEST_DIR / name))
print('\nSample camera_urls.jsonl rows:')
for row in iter_jsonl(HARVEST_DIR / 'camera_urls.jsonl', limit=5):
    print(json.dumps(row, indent=2)[:2000])


## Step 2 — Run validation from HLS handoff

This cell intentionally uses a visible `!camera-discovery run ...` shell command so progress output streams in the notebook.

For interactive notebook runs over large handoff sets, this notebook defaults to `balanced` validation with `--http-timeout 10`. `balanced` validates HLS playlists without the extra full-profile segment check, so it is much faster for thousands of candidates. Use `full` only when you intentionally want additional segment checks.

Validation is parallel and bounded by worker count, not by candidate count. No validation candidate cap is applied.

If the optional/advisory geocoder-referee LLM is rate-limited, the source should write `geocoder_referee_llm_error.json` and continue with deterministic geocoder scores.


In [6]:
%%time
RUN_PROFILE = "balanced"
HTTP_TIMEOUT_SECONDS = 10
RUN_DIR = Path('runs/run-from-harvest-hls-balanced')
RERUN_PIPELINE = False
expected_run = [RUN_DIR / 'logs' / 'run_summary.json', RUN_DIR / 'logs' / 'candidate_discovery_summary.json']
if RERUN_PIPELINE and RUN_DIR.exists():
    shutil.rmtree(RUN_DIR)

if not all(path.exists() for path in expected_run):
    print('Running visible CLI command. Output will stream below.')
    print('Validation is parallel and bounded by worker count; no validation candidate cap is applied.')
    print(f'Effective notebook profile: {RUN_PROFILE}; HTTP timeout: {HTTP_TIMEOUT_SECONDS}s')
    !camera-discovery run "{QUERY}" \
    --profile "{RUN_PROFILE}" \
    --output-dir "{RUN_DIR}" \
    --harvest-input "{HARVEST_DIR / 'harvest_handoff.json'}" \
    --harvest-input-mode handoff-only \
    --browser-backend "{BROWSER_BACKEND}" \
    --http-timeout "{HTTP_TIMEOUT_SECONDS}" \
    --progress-style plain
else:
    print('Skipping pipeline: completion artifacts already exist. Set RERUN_PIPELINE=True to rerun.')


Running visible CLI command. Output will stream below.
Validation is parallel and bounded by worker count; no validation candidate cap is applied.
Effective notebook profile: balanced; HTTP timeout: 10s
Pipeline mode: normal discovery pipeline
Query: Virginia traffic cameras
Output dir: runs/run-from-harvest-hls-balanced
Profile: balanced
Validation enabled: True (trusted outputs require validation)
LLM provider: ollama-cloud
Target-intent model: gemma3:12b-cloud
Discovery mode: both
Sources file: /content/camera-discovery/SOURCES.md (exists=True)
Browser capture: enabled=False backend=playwright
Harvest input: runs/harvest-california-hls/harvest_handoff.json (exists=True)
Harvest input mode: handoff-only
Harvest handoff filter: ['.m3u8'] default=filtered_media_records 
artifact=camera_urls_jsonl
Harvest input trust: source-provided, unvalidated, untrusted seed data
Progress: resolving targets...
Progress: resolved 1 target(s).
Targets resolved: 1
  - virginia: Virginia | geometry=veri

## Inspect full-validation pipeline outputs

In [ ]:
# Inspect pipeline/run outputs.
RUN_DIR = Path(RUN_DIR)
print('Run directory:', RUN_DIR)
list_existing([
    RUN_DIR / 'logs' / 'run_summary.json',
    RUN_DIR / 'logs' / 'run_explanation.json',
    RUN_DIR / 'logs' / 'candidate_discovery_summary.json',
    RUN_DIR / 'logs' / 'candidate_priority_summary.json',
    RUN_DIR / 'logs' / 'validation_summary.json',
    RUN_DIR / 'logs' / 'validation_priority_summary.json',
    RUN_DIR / 'camera_candidates_table.csv',
    RUN_DIR / 'untrusted_camera_candidates.geojson',
    RUN_DIR / 'camera.geojson',
    RUN_DIR / 'map.html',
    RUN_DIR / 'review_artifacts.zip',
])

print('\nRun summary:')
print_json(RUN_DIR / 'logs' / 'run_summary.json')
print('\nCandidate discovery summary:')
print_json(RUN_DIR / 'logs' / 'candidate_discovery_summary.json')
print('\nCandidate priority summary:')
print_json(RUN_DIR / 'logs' / 'candidate_priority_summary.json')
print('\nValidation summary:')
print_json(RUN_DIR / 'logs' / 'validation_summary.json')

# Inspect CSV quickly without pandas.
csv_path = RUN_DIR / 'camera_candidates_table.csv'
if csv_path.exists():
    import csv
    with csv_path.open(newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        rows = list(reader)
    print('\nCandidate table rows:', len(rows))
    print('Media counts:', dict(Counter(row.get('camera_type') or row.get('media_type') or '(missing)' for row in rows)))
    print('Scope counts:', dict(Counter(row.get('scope_status') or '(missing)' for row in rows)))
    print('Priority buckets:', dict(Counter(row.get('candidate_priority_bucket') or '(missing)' for row in rows)))
    print('\nFirst 5 candidate table rows:')
    for row in rows[:5]:
        print({k: row.get(k) for k in ['camera_type', 'scope_status', 'candidate_priority_bucket', 'stream_url', 'latitude', 'longitude', 'validation_status'] if k in row})
else:
    print('No candidate table found.')

# GeoJSON feature counts.
for geojson_name in ['camera.geojson', 'untrusted_camera_candidates.geojson']:
    path = RUN_DIR / geojson_name
    data = read_json(path)
    if data:
        features = data.get('features', [])
        print(f"{geojson_name}: {len(features)} features")
        print('Feature scope counts:', dict(Counter((feat.get('properties') or {}).get('scope_status') or '(missing)' for feat in features)))


## Handoff-specific assertions to review

In [ ]:
# Manual/diagnostic checks: HLS-only handoff should not inject image snapshots by default.
summary = read_json(RUN_DIR / 'logs' / 'candidate_discovery_summary.json') or {}
validation = read_json(RUN_DIR / 'logs' / 'validation_summary.json') or {}
print('Harvest-input media summary:', summary.get('harvest_input_media_counts') or summary.get('harvest_input', {}).get('media_counts'))
print('Combined media summary:', summary.get('combined_media_counts') or summary.get('media_counts'))
print('Validation attempted/skipped:', validation.get('attempted'), validation.get('skipped'))
print('Trusted camera.geojson exists:', (RUN_DIR / 'camera.geojson').exists())


## Package outputs

In [7]:
# Optional: package and download outputs.
# Run this after the workflow completes.
package_output('runs/run-from-harvest-hls-balanced')


Created archive: /content/runs_run-from-harvest-hls-balanced.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

'/content/runs_run-from-harvest-hls-balanced.zip'